# CUAD Fine-Tuning Notebook

This notebook is the local-first workspace for environment bootstrap, first CUAD inspection, preprocessing design, prompt and schema experiments, and later metrics review.

Current scope:
- runnable environment inspection cells
- safe CUAD file discovery and first inspection cells
- no training or heavy model downloads yet

## Environment Inspection

These cells are safe to run locally even if there is no GPU, no CUAD dataset, and no FABRIC-mounted storage.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import platform
import shutil
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
FABRIC_PATHS = {
    "data": Path("/mnt/project/data"),
    "checkpoints": Path("/mnt/project/checkpoints"),
    "cache": Path("/mnt/project/cache/hf"),
    "artifacts": Path("/mnt/project/artifacts"),
}

print(f"project_root={PROJECT_ROOT}")

In [ ]:
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory exists:", PROJECT_ROOT.exists())

usage = shutil.disk_usage(PROJECT_ROOT)
print("Free disk space (GB):", round(usage.free / (1024 ** 3), 2))

In [ ]:
packages = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "evaluate",
    "numpy",
    "pandas",
    "sklearn",
    "matplotlib",
]

availability = {
    package: importlib.util.find_spec(package) is not None for package in packages
}
availability

In [ ]:
try:
    import torch

    print("torch_version=", torch.__version__)
    print("cuda_available=", torch.cuda.is_available())
    if torch.cuda.is_available():
        device_index = torch.cuda.current_device()
        print("cuda_device=", torch.cuda.get_device_name(device_index))
        props = torch.cuda.get_device_properties(device_index)
        print("vram_gb=", round(props.total_memory / (1024 ** 3), 2))
    else:
        print("No CUDA device detected. Local CPU-only inspection is still fine.")
except Exception as exc:
    print("Torch inspection skipped:", exc)

In [ ]:
hf_env = {
    "HF_HOME": os.environ.get("HF_HOME"),
    "TRANSFORMERS_CACHE": os.environ.get("TRANSFORMERS_CACHE"),
}
print(hf_env)

for name, path in FABRIC_PATHS.items():
    print({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "writable": os.access(path, os.W_OK) if path.exists() else False,
    })

In [ ]:
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    result = subprocess.run(
        [nvidia_smi, "--query-gpu=driver_version,name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode == 0:
        print("nvidia_smi_detected=True")
        print(result.stdout.strip() or "Driver query returned no rows.")
    else:
        print("nvidia_smi_detected=True but driver query failed:")
        print(result.stderr.strip() or result.stdout.strip() or f"exit_code={result.returncode}")
else:
    print("nvidia-smi not found on PATH. Re-run this check on the FABRIC VM during bootstrap.")

print("Model download verification is deferred to the FABRIC VM bootstrap phase.")

## CUAD Inspection

Use this section to inspect likely CUAD inputs before locking the preprocessing path.

Expected input locations:
- local: `./data`, `./datasets`, `./cuad`, `./CUAD`
- FABRIC: `/mnt/project/data`, `/mnt/project/data/cuad`, `/mnt/project/data/CUAD`

Format notes:
- **Wide/master format** keeps more of the source annotation layout in one record or one contract-centric table. It is useful for inspection and traceability, but it is awkward as the direct training format for structured generation.
- **QA-style format** reshapes the task into one contract/category example per row or record, usually with fields like `question`, `context`, `answers`, or similar. This is usually the cleaner working format for training and evaluation.
- **No-answer cases** must stay explicit. Do not drop them during inspection or later preprocessing; they are part of the task definition and should map cleanly to `found=false` with null answer/evidence fields in the structured target schema.
- **Contract-level splits** are required. Any later long-format expansion should happen only after train/validation/test membership is fixed by contract, not by chunk or example row.

In [ ]:
candidate_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "cuad",
    PROJECT_ROOT / "CUAD",
    FABRIC_PATHS["data"],
    FABRIC_PATHS["data"] / "cuad",
    FABRIC_PATHS["data"] / "CUAD",
]

print("Candidate CUAD roots:")
for path in candidate_roots:
    print({"path": str(path), "exists": path.exists()})

existing_roots = [path for path in candidate_roots if path.exists()]

In [ ]:
interesting_suffixes = {".json", ".jsonl", ".csv", ".parquet", ".txt"}
discovered_files = []

for root in existing_roots:
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.suffix.lower() in interesting_suffixes:
            discovered_files.append(path)
        if len(discovered_files) >= 40:
            break
    if len(discovered_files) >= 40:
        break

if discovered_files:
    print("Discovered candidate files:")
    for path in discovered_files:
        print(path)
else:
    print("CUAD files not found yet. Place the dataset in one of the expected locations, then rerun this section.")

In [ ]:
import csv
import json
from itertools import islice


def infer_format_from_fields(field_names):
    lowered = {str(name).strip().lower() for name in field_names if name is not None}
    qa_markers = {"question", "context", "answers", "answer_start", "answer_text", "id"}
    structured_markers = {"category", "found", "normalized_answer", "evidence_text"}
    if len(qa_markers & lowered) >= 2:
        return "qa-style-like"
    if len(structured_markers & lowered) >= 2:
        return "structured-output-like"
    if len(lowered) >= 12:
        return "wide/master-like"
    return "unclear"


def summarize_candidate(path):
    suffix = path.suffix.lower()
    summary = {
        "path": str(path),
        "suffix": suffix,
        "likely_format": "unclear",
        "schema_keys": None,
    }

    if suffix == ".csv":
        with path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            fields = reader.fieldnames or []
        summary["schema_keys"] = fields[:20]
        summary["likely_format"] = infer_format_from_fields(fields)
        return summary

    if suffix == ".jsonl":
        with path.open("r", encoding="utf-8") as handle:
            first_line = next((line.strip() for line in handle if line.strip()), None)
        if first_line:
            record = json.loads(first_line)
            fields = list(record.keys()) if isinstance(record, dict) else []
            summary["schema_keys"] = fields[:20]
            summary["likely_format"] = infer_format_from_fields(fields)
        return summary

    if suffix == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        sample = payload
        if isinstance(payload, list):
            sample = payload[0] if payload else {}
        elif isinstance(payload, dict):
            for key in ("data", "examples", "records", "contracts"):
                value = payload.get(key)
                if isinstance(value, list) and value:
                    sample = value[0]
                    break
        fields = list(sample.keys()) if isinstance(sample, dict) else []
        summary["schema_keys"] = fields[:20]
        summary["likely_format"] = infer_format_from_fields(fields)
        return summary

    if suffix == ".parquet":
        try:
            import pandas as pd

            frame = pd.read_parquet(path)
            fields = frame.columns.tolist()
            summary["schema_keys"] = fields[:20]
            summary["likely_format"] = infer_format_from_fields(fields)
        except Exception as exc:
            summary["schema_keys"] = [f"parquet_preview_unavailable: {exc}"]
        return summary

    if suffix == ".txt":
        summary["schema_keys"] = ["raw_text_document"]
        summary["likely_format"] = "source-text-only"
        return summary

    return summary


schema_summaries = []
for path in discovered_files[:10]:
    summary = summarize_candidate(path)
    schema_summaries.append(summary)
    print(summary)

if not schema_summaries:
    print("Schema summary skipped: no candidate CUAD files found yet.")

In [ ]:
example_preview_path = discovered_files[0] if discovered_files else None
if example_preview_path is None:
    print("Example record preview skipped: no CUAD candidate files found yet.")
else:
    print(f"Previewing: {example_preview_path}")
    suffix = example_preview_path.suffix.lower()
    if suffix == ".json":
        payload = json.loads(example_preview_path.read_text(encoding="utf-8"))
        if isinstance(payload, list):
            print(payload[0] if payload else "JSON list is empty.")
        elif isinstance(payload, dict):
            sample = payload
            for key in ("data", "examples", "records", "contracts"):
                value = payload.get(key)
                if isinstance(value, list) and value:
                    sample = value[0]
                    break
            print(sample)
        else:
            print(payload)
    elif suffix == ".jsonl":
        with example_preview_path.open("r", encoding="utf-8") as handle:
            first_line = next((line.strip() for line in handle if line.strip()), None)
        print(json.loads(first_line) if first_line else "JSONL file is empty.")
    elif suffix == ".csv":
        with example_preview_path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            print(next(reader, "CSV file has no data rows."))
    elif suffix == ".parquet":
        try:
            import pandas as pd

            frame = pd.read_parquet(example_preview_path)
            if frame.empty:
                print("Parquet file has no rows.")
            else:
                print(frame.head(1).to_dict(orient="records")[0])
        except Exception as exc:
            print(f"Parquet preview unavailable: {exc}")
    else:
        with example_preview_path.open("r", encoding="utf-8", errors="replace") as handle:
            preview_lines = list(islice(handle, 5))
        print("".join(preview_lines) if preview_lines else "Text-like file is empty.")

### Primary V1 Working-Format Recommendation

Prefer a **QA-style long working format** for v1, derived from the raw CUAD source after inspection. Keep one contract/category example per row or record, preserve a stable contract identifier for contract-level splits, and keep no-answer cases explicit. Treat wide/master source files as inspection and traceability inputs rather than the direct training format.

## Structured-Generation Example Prototype

This section prototypes the v1 long-form training record design before any real preprocessing code is written.

Chunking assumptions for the prototype:
- each training row is one `(contract_id, chunk_id, category)` example
- chunking happens over contract text with overlap, but retrieval remains out of scope
- a positive row requires evidence text to be visible inside the selected chunk
- overlapping chunks may create multiple positive rows for the same contract/category if the evidence spans multiple windows
- no-answer rows remain explicit negatives and are not filtered out

In [ ]:
from pprint import pprint

prototype_chunk_examples = [
    {
        "contract_id": "contract_001",
        "chunk_id": "contract_001_chunk_000",
        "chunk_index": 0,
        "category": "Assignment",
        "found": True,
        "chunk_text": "Neither party may assign this Agreement without the prior written consent of the other party.",
        "normalized_answer": "prior written consent required",
        "evidence_text": "Neither party may assign this Agreement without the prior written consent of the other party.",
    },
    {
        "contract_id": "contract_001",
        "chunk_id": "contract_001_chunk_001",
        "chunk_index": 1,
        "category": "Change of Control",
        "found": False,
        "chunk_text": "This section covers notice procedures and payment timing only.",
        "normalized_answer": None,
        "evidence_text": None,
    },
    {
        "contract_id": "contract_002",
        "chunk_id": "contract_002_chunk_000",
        "chunk_index": 0,
        "category": "Governing Law",
        "found": True,
        "chunk_text": "This Agreement is governed by the laws of the State of New York, excluding conflict-of-law rules.",
        "normalized_answer": "New York",
        "evidence_text": "governed by the laws of the State of New York",
    },
    {
        "contract_id": "contract_002",
        "chunk_id": "contract_002_chunk_001",
        "chunk_index": 1,
        "category": "Termination for Convenience",
        "found": True,
        "chunk_text": "The customer may terminate this Agreement for convenience on thirty days' written notice.",
        "normalized_answer": "30 days written notice",
        "evidence_text": "terminate this Agreement for convenience on thirty days' written notice",
    },
    {
        "contract_id": "contract_003",
        "chunk_id": "contract_003_chunk_000",
        "chunk_index": 0,
        "category": "Exclusivity",
        "found": False,
        "chunk_text": "The distributor is free to market competing products during the term.",
        "normalized_answer": None,
        "evidence_text": None,
    },
    {
        "contract_id": "contract_003",
        "chunk_id": "contract_003_chunk_001",
        "chunk_index": 1,
        "category": "Audit Rights",
        "found": True,
        "chunk_text": "Licensor may audit Licensee's records once per year upon ten business days' notice.",
        "normalized_answer": "once per year with 10 business days notice",
        "evidence_text": "audit Licensee's records once per year upon ten business days' notice",
    },
    {
        "contract_id": "contract_004",
        "chunk_id": "contract_004_chunk_000",
        "chunk_index": 0,
        "category": "Non-Compete",
        "found": True,
        "chunk_text": "Employee agrees that for twelve months after termination, Employee will not compete within Illinois.",
        "normalized_answer": "12 months in Illinois",
        "evidence_text": "for twelve months after termination, Employee will not compete within Illinois",
    },
    {
        "contract_id": "contract_004",
        "chunk_id": "contract_004_chunk_001",
        "chunk_index": 1,
        "category": "Most Favored Nation",
        "found": False,
        "chunk_text": "This chunk describes invoicing, taxes, and payment remittance details.",
        "normalized_answer": None,
        "evidence_text": None,
    },
]

len(prototype_chunk_examples)

In [ ]:
def build_structured_target(category, found, normalized_answer, evidence_text):
    if not found and (normalized_answer is not None or evidence_text is not None):
        raise ValueError("No-answer examples must use null answer and evidence fields.")
    if found and (not isinstance(evidence_text, str) or not evidence_text.strip()):
        raise ValueError("Positive examples must include non-empty evidence text.")
    return {
        "category": category,
        "found": found,
        "normalized_answer": normalized_answer if found else None,
        "evidence_text": evidence_text if found else None,
    }


def build_instruction_text(category):
    return (
        "Read the contract chunk and return JSON only. Use exactly these keys: "
        "category, found, normalized_answer, evidence_text. For the CUAD category "
        f"'{category}', if the category is absent set found=false, "
        "normalized_answer=null, and evidence_text=null."
    )


def prototype_to_training_record(example):
    target = build_structured_target(
        category=example["category"],
        found=example["found"],
        normalized_answer=example["normalized_answer"],
        evidence_text=example["evidence_text"],
    )
    return {
        "contract_id": example["contract_id"],
        "chunk_id": example["chunk_id"],
        "chunk_index": example["chunk_index"],
        "category": example["category"],
        "instruction": build_instruction_text(example["category"]),
        "input_text": example["chunk_text"],
        "target_json": json.dumps(target, ensure_ascii=True),
        "target_structured": target,
    }


prototype_training_records = [
    prototype_to_training_record(example) for example in prototype_chunk_examples
]

for record in prototype_training_records:
    pprint(record)
    print("-" * 80)

In [ ]:
no_answer_count = sum(not record["target_structured"]["found"] for record in prototype_training_records)
positive_count = len(prototype_training_records) - no_answer_count
summary = {
    "num_records": len(prototype_training_records),
    "num_positive": positive_count,
    "num_no_answer": no_answer_count,
    "example_columns": list(prototype_training_records[0].keys()),
}
summary

### Prototype Design Notes

- The long-form working table uses one row per `(contract_id, chunk_id, category)`.
- `input_text` holds only the selected chunk, not the full contract.
- `target_json` is the supervised text target for structured generation, while `target_structured` is a notebook-only convenience view for debugging.
- No-answer rows remain in the same table with `found=false` and null answer/evidence fields.
- Retrieval stays out of scope: the prototype assumes chunk selection is a preprocessing step, not a retrieval step.

## Preprocessing Design Placeholder

Planned first-pass decisions:
- inspect the raw CUAD source format before fixing the working schema
- keep splits at the contract level
- target structured generation with the canonical output schema

In [ ]:
target_output_schema = {
    "category": "<CUAD category>",
    "found": True,
    "normalized_answer": "<short normalized answer or null>",
    "evidence_text": "<supporting contract text or null>",
}
target_output_schema

## Prompt and Output Experiments Placeholder

Use this section later for prompt drafts, schema validation checks, and small manual examples.

## Metrics Review Placeholder

Use this section later for category-level metrics, sample predictions, and qualitative error review.